# Śledzenie obiektów

<img src="https://i.imgur.com/wKXXFkQ.png" width="500">

## Wstęp
W erze cyfrowej, w obliczu rosnącej lawinowo ilości danych wideo, zdolność do ich automatycznego rozpoznawania i interpretowania staje się kluczowa w wielu dziedzinach – od bezpieczeństwa publicznego po autonomiczne pojazdy. Technologie oparte na głębokim uczeniu rewolucjonizują sposób, w jaki przetwarzamy informacje wizualne. Kluczowym wyzwaniem jest tu detekcja i śledzenie obiektów na filmach wideo.

Celem tego zadania jest opracowanie algorytmu, który będzie w stanie analizować sekwencje ruchów w grze "trzy kubki". Uczestnicy mają za zadanie określić końcową pozycję kubków po serii ruchów, korzystając z analizy statycznych obrazów z każdej klatki nagrania.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

# Poniższe funkcje ułatwiają pracę z dostarczonymi danymi
# W kolejnych komórkach zobaczysz przykłady ich użycia
from utils.utils import get_level_info, get_video_data, display_video, download_and_replace_data

FINAL_EVALUATION_MODE = False
# W czasie sprawdzania Twojego rozwiązania, zmienimy tę wartość na True
# Wartość tej flagi M U S I zostać ustawiona na False w rozwiązaniu, które nam nadeślesz!

images, coordinates, target, path_to_images = get_video_data(level=2,video_id=0,dataset="example")
display_video(images,rescale=0.7,FINAL_EVALUATION_MODE=FINAL_EVALUATION_MODE)

## Zadanie 2: Okluzje, rozmycia, przesłonienie

Celem zadania jest opracowanie algorytmu, który potrafi przetwarzać sekwencje obrazów z gry w trzy kubki, nawet gdy występują rozmycia czy przesłonięcia. Zadanie ma na celu nauczenie maszyny wykorzystywania ciągłości informacji z kolejnych klatek, aby mimo chwilowych utrudnień w percepcji, mogła skutecznie określić końcową pozycję kubków.

Napisz algorytm który poradzi sobie z trudnieszym zbiorem danych - `Level_2`. Składa się on z animacji, w których pojawiają się dodatkowe utrudnienia:
- Przesłonięcia obiektu przez inny, powodujące, że są one traktowane jako jeden,
- Prostokąty ograniczające nie są już idealnie dopasowane do obiektów,
- Prostokąty ograniczające nie są widoczne we wszystkich klatkach.

Będziesz miał dostęp zarówno do wszystkich klatek animacji, jak i do oznaczonych przez nas prostokątów ograniczających, w których znajdują się kubki. Co ważne, algorytm, który będziesz tworzył ma korzystać jedynie z informacji o prostokątach ograniczających. W tym zadaniu, klatki wideo są dostarczone jedynie do wizualizacji przykładów i algorytmu, na własne potrzeby.

Punkty za to zadanie będą przyznane za osiągnięcie jak najdokładniejszych predykcji na zbiorze testowym. Kryterium będzie *accuracy* i spodziewamy się wyników powyżej `80%`. Ewaluacja na zbiorze testowym będzie dokonana przez organizatorów.

## Pliki zgłoszeniowe
Tylko ten notebook zawierający **kod** oraz **krótki raport** opisujący Twoje rozwiązanie (do 300 słów). Miejsce na raport znajdziesz na końcu tego notebooka.

## Ograniczenia
- Twoja funkcja powinna zwracać predykcje w maksymalnie 5 minut używając Google Colab bez GPU.

## Uwagi i wskazówki
- Testuj swoje rozwiązanie na zbiorze plików wideo `level_2`.
- **Skuteczność modelu**: przetestuj skuteczność modelu na zbiorze walidacyjnym używając dostarczonej przez nas funkcji **submission_script**, umieść ten wynik w raporcie.

## Ewaluacja
Pamiętaj, że podczas sprawdzania flaga `FINAL_EVALUATION_MODE` zostanie ustawiona na `True`. Za pomocą skryptu `validation_script.py` możesz upewnić się, że Twoje rozwiązanie zostanie prawidłowo wykonane na naszych serwerach oceniających.

Za to podzadanie możesz zdobyć pomiędzy 0 i 0.5 punktów. Zdobędziesz 0 punktów jeśli Twoje accuracy na zbiorze testowym będzie poniżej 50%. Jeśli będzie większe niż 95%, otrzymasz 0.5 punktu. Pomiędzy tymi wartościami, wynik rośnie liniowo z wartością metryki.

# Kod startowy

In [ ]:
# Poniższe biblioteki są wystarczające do wykonania wszystkich zadań
# Jeśli jednak chcesz użyć innych, sprawdź czy są dostępne na serwerze (requirements.txt)
import numpy as np
import os
import matplotlib.pyplot as plt
import torch
import IPython.display
import json
import PIL
import sklearn as sk

In [ ]:
# funkcja pomocnicza do ładowania danych
images, _, _, _ = get_video_data(level=2,video_id=0,dataset="example")

with open(os.path.join(os.getcwd(),'example_tracks','tracks_2_0.json'), 'r') as f:
    tracks = json.load(f)

for key in tracks.keys():
    tracks[key] = [tuple(el) for el in tracks[key]]

# funkcja pomocnicza do wyświetlania danych
display_video(images,
                tracks=tracks,
                rescale=0.7,
                FINAL_EVALUATION_MODE=FINAL_EVALUATION_MODE)

In [ ]:
# Pobieranie danych do podzadań 1, 2 i 3 (około ~646Mb), skrypt będzie wykonywał się parę minut
# Wystarczy, że pobierzesz dane tylko raz. Na serwerze sprawdzającym dane będą już pobrane,
# struktura plików będzie identyczna jak tutaj
if not FINAL_EVALUATION_MODE:
    download_and_replace_data()

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# funkcja pomocnicza do testowania algorytmu
def submission_script(algorithm,level,verbose=False,dataset="valid"):
    num_videos, _ = get_level_info(level=level,dataset=dataset)
    correct = []
    exception_messages = set()
    for video_number in range(num_videos):
        _, coordinates, target, _ = get_video_data(level=level,video_id=video_number,dataset=dataset)
        try:
            prediction = algorithm(coordinates)
            if tuple(target) == tuple(prediction):
                correct.append(1)
            else:
                correct.append(0)
            if verbose:
                print(f"Video: animation_{str(video_number).zfill(4)}")
                print(f"Prediction: {prediction}")
                print(f"Target:     {target}")
                print(f"Score: {tuple(target) == tuple(prediction)}", end='\n\n')
        except Exception as e:
            correct.append(0)
            exception_messages.add(str(e))
    if verbose:
        print(f"Accuracy: {np.mean(correct)}")
        print(f"Correctness: {correct}")
    return np.sum(correct) / num_videos, correct, exception_messages

# Twoje rozwiązanie

In [ ]:
from scipy.optimize import linear_sum_assignment

class KalmanFilter:
    def __init__(self, F=None, B=None, H=None, Q=None, R=None, P=None, x=None):
        self.n = x.shape[0]
        self.F = F if F is not None else np.eye(self.n)
        self.B = B if B is not None else np.zeros((self.n, 1))
        self.H = H if H is not None else np.eye(self.n)
        self.Q = Q if Q is not None else np.eye(self.n)
        self.R = R if R is not None else np.eye(self.n)
        self.P = P if P is not None else np.eye(self.n)
        self.x = x

    def predict(self, u=0):
        self.x = np.dot(self.F, self.x) + np.dot(self.B, u)
        self.P = np.dot(np.dot(self.F, self.P), self.F.T) + self.Q
        return self.x

    def update(self, z):
        y = z - np.dot(self.H, self.x)
        S = np.dot(self.H, np.dot(self.P, self.H.T)) + self.R
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))
        self.x = self.x + np.dot(K, y)
        I = np.eye(self.n)
        self.P = np.dot((I - np.dot(K, self.H)), self.P)
        return self.x


def your_algorithm_task_2(coordinates):
    dt = 3.2
    F = np.array([[1, dt],
                  [0, 1]])
    P = np.array([[1000, 0],
                  [0, 1000]])
    H = np.array([[1, 0]])
    Q = np.array([[1, 0],
                  [0, 3]]) * 0.8
    R = np.array([[10]]) * 2

    frames = [coordinates[x] for x in sorted(coordinates.keys())]
    start_pos = np.array(frames[0])

    kfs = []
    for i in range(3):
      kfs.append([])
      for j in range(4):
        kfs[-1].append(KalmanFilter(F=F.copy(), H=H.copy(), Q=Q.copy(), R=R.copy(), P=P.copy(), x=np.array([[start_pos[i,j]],[0]])))

    ord = np.argsort(start_pos[:, 0])
    pos = np.arange(3)
    pos[ord] = np.arange(3)
    pred_pos = np.zeros_like(start_pos)


    for idx, frame in enumerate(frames[1:]):
      for i in range(3):
        for j in range(4):
          pred_pos[i, j] = kfs[i][j].predict()[0][0]

      if frame:
        costs = np.zeros((3, len(frame)))
        for row in range(3):
          for col in range(len(frame)):
            costs[row, col] += np.sum((pred_pos[row] - np.array(frame[col]))**2)
        rows, cols = linear_sum_assignment(costs, maximize=False)
        pred_pos[rows] = np.array(frame)[cols]

      start_pos = pred_pos
      for i in range(3):
        for j in range(4):
          kfs[i][j].update(np.array([[start_pos[i, j]]]))


    return list(pos[np.argsort(start_pos[:, 0])])

In [ ]:
# Sprawdź jak działa Twój algorytm
accuracy, correctness, _ = submission_script(
    algorithm=your_algorithm_task_2,
    level=2,
    verbose=True,
    dataset="train")

In [ ]:
# zapisz swój raport do zmiennej poniżej, abyśmy mogli go później automatycznie odczytać sprawdzaczką
raport_2 = \
"""
Raport z zadania:
Prowadzimy analogiczne rozumowanie jak w podzadaniu 1.
Aby poradzić sobie ze znikającymi bounding boxami (oraz z filtrowaniem szumu) korzystam z filtru Kalmana.
Jest to algorytm który na podstawie poprzednich pomiarów przewiduje stan systemu w przyszłości. Posłużyłem się modelem stałej prędkości.
Dla każdej klatki przewiduję gdzie znajdą się bounding boxy w następnej klatce. Następnie znajduję minimalne przyporządkowanie między przewidywaniami a faktycznymi bounding boxami (których może być mniej niż 3).
Predykcje którym zostały przyporzadkowane boxy aktualizuję, a pozostałe pozostawiam niezmienione.
Zakładam że takie zaktualizowane predykcje są miejscami boxów w nastepnej klatce.
"""